# Phân tích SQL - Dữ liệu thương mại điện tử Olist

Notebook này chạy 8 câu truy vấn Q01–Q08 từ tệp `sql/03_queries.sql` bằng SQLite và pandas.

### Mục tiêu phân tích
- Đánh giá phân bố và mức cân bằng của `review_score`.
- So sánh giao trễ, phí vận chuyển, độ phức tạp đơn hàng, thanh toán và khu vực với mức hài lòng.


## Tóm tắt và tổng hợp notebook

### Mục đích
Notebook phân tích dữ liệu Brazilian E-Commerce Public Dataset by Olist giai đoạn 2016–2018 nhằm trả lời câu hỏi trọng tâm:
- Những yếu tố nào trong quá trình mua và giao hàng chi phối mức độ hài lòng `review_score`?

### Mục tiêu
Notebook kết nối SQLite với `data/olist.db`, tải view và đặc trưng từ `sql/03_queries.sql`, đồng thời tổng hợp item, thanh toán và người bán theo từng đơn hàng trước khi phân tích để tránh nhân đôi dữ liệu.

8 truy vấn được tổ chức thành các nhóm mục tiêu:
- **Q01:** Phân bố điểm đánh giá và mức mất cân bằng lớp.
- **Q02:** Mối quan hệ giữa giao trễ và điểm review.
- **Q03:** Phân tích mức độ trễ theo các bucket thời gian.
- **Q04:** Tổng phí ship và tỷ lệ phí ship trên giá trị hàng hóa.
- **Q05:** Độ phức tạp qua số sản phẩm, số mặt hàng và số người bán trong đơn.
- **Q06:** Phương thức thanh toán và số kỳ trả góp.
- **Q07:** Xu hướng review theo thời gian 22 tháng.
- **Q08:** Phân hóa mức hài lòng theo bang và khu vực khách hàng.

### Đầu ra và cách sử dụng
- Mỗi truy vấn trả về một `DataFrame`, dùng cho so sánh, trực quan hóa và rút kết luận kinh doanh/logistics.
- `v_model_features_at_purchase` chỉ chứa thông tin có tại thời điểm đặt hàng và phù hợp cho mô hình dự đoán sớm.
- `v_model_features_post_delivery` bổ sung biến giao hàng và chỉ phù hợp khi dự đoán sau khi đơn hàng đã được giao.
- Việc tách hai tập feature giúp kiểm soát data leakage và hỗ trợ câu hỏi nghiên cứu thứ hai một cách minh bạch.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

current_dir = Path.cwd().resolve()
base_dir = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / "scripts" / "database" / "olist.db").exists()),
    None
)

if base_dir is None:
    raise FileNotFoundError("Không tìm thấy scripts/database/olist.db trong thư mục hiện tại hoặc các thư mục cha")

db_path = base_dir / "scripts" / "database" / "olist.db"
sql_path = base_dir / "sql" / "03_queries.sql"

if not db_path.exists():
    raise FileNotFoundError(f"Khong tim thay database: {db_path}")

print("Database:", db_path)
print("SQL source:", sql_path)

Database: D:\FPT UNIVERSITY\Kì 3\ADY201m\ADY291c-AI2107-Group2\scripts\database\olist.db
SQL source: D:\FPT UNIVERSITY\Kì 3\ADY201m\ADY291c-AI2107-Group2\sql\03_queries.sql


In [2]:
# Tạo các view cần thiết; 
# Mỗi câu truy vấn phân tích được thực thi riêng ở cell bên dưới.
sql_text = sql_path.read_text(encoding="utf-8")
q01_marker = "-- Q1:"
leakage_marker = "-- LEAKAGE-SAFE MODEL FEATURE VIEWS"

view_setup = sql_text.split(q01_marker, 1)[0]
if leakage_marker in sql_text:
    view_setup += "\n" + leakage_marker + "\n"
    view_setup += sql_text.split(leakage_marker, 1)[1]

conn = sqlite3.connect(db_path, timeout=30)
conn.execute("PRAGMA cache_size = -64000")
conn.execute("PRAGMA temp_store = MEMORY")
conn.executescript(view_setup)

views = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
).fetchall()

print("Danh sách view:")
for view_name, in views:
    print("-", view_name)

print("Số đơn phân tích:", conn.execute(
    "SELECT COUNT(*) FROM v_review_order_features"
).fetchone()[0])

Danh sách view:
- v_model_features_at_purchase
- v_model_features_post_delivery
- v_review_order_features
Số đơn phân tích: 95329


## Quy ước dữ liệu

View `v_review_order_features` chứa một dòng cho mỗi đơn đã giao và đã có đánh giá. Dữ liệu sản phẩm và thanh toán được tổng hợp theo `order_id` trước khi JOIN để tránh nhân đôi đánh giá.

- Nếu một đơn có nhiều review, phân tích dùng review có thời điểm `review_creation_date` mới nhất; `order_id` chỉ giữ một dòng trong view phân tích.
- Đơn không có review hoặc không thuộc trạng thái đã giao được loại khỏi `v_review_order_features` vì view này phục vụ phân tích mức hài lòng sau giao hàng.
- `delay_days <= 0`: đơn giao đúng hạn hoặc sớm hơn dự kiến.
- `delay_days > 0`: đơn giao trễ so với ngày dự kiến.
- `review_score`: điểm đánh giá từ 1 đến 5.


## Q01: Phân bố điểm đánh giá và mức mất cân bằng lớp

**Mục tiêu:** Xem số lượng và tỷ lệ đơn theo từng mức `review_score` từ 1 đến 5.

**Cách thực hiện:** Nhóm dữ liệu bằng `GROUP BY review_score`, đếm số đơn và tính phần trăm của mỗi điểm so với tổng số đơn.

**Ý nghĩa đối với mô hình:** Kết quả cho biết phân bố của biến mục tiêu `review_score`. Điều này giúp nhận diện mức mất cân bằng lớp, tránh đánh giá mô hình chỉ bằng accuracy và xác định có nên giữ bài toán hồi quy hoặc chuyển thành phân loại 3 nhóm/nhiều lớp.

**Mục đích:** Định hướng thiết kế nhãn, metric và chiến lược xử lý lớp thiểu số trước khi huấn luyện mô hình dự đoán điểm đánh giá.

**Kết quả đầu ra:** DataFrame `df_q01` chứa `review_score`, số đơn và tỷ lệ phần trăm.

In [3]:
q01 = """
SELECT review_score,
       COUNT(*) AS order_count,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM v_review_order_features), 2) AS percentage
FROM v_review_order_features
GROUP BY review_score
ORDER BY review_score;
"""

df_q01 = pd.read_sql_query(q01, conn)
df_q01


,review_score,order_count,percentage
0,1,9277,9.73
1,2,2900,3.04
2,3,7865,8.25
3,4,18807,19.73
4,5,56480,59.25


## Kết luận và bước tiếp theo — Q01

**Kết luận**

- Phân bố `review_score` lệch mạnh: 5 sao chiếm **59,25%**, 4 sao **19,73%**, 1–2 sao chiếm **12,77%** trong **95.329 đơn đã giao có review**.
- Nếu mô hình đoán mọi đơn đều là 5 sao, accuracy sẽ khoảng **59,25%**. Accuracy cao như vậy không chứng minh mô hình nhận diện tốt các đơn 1–2 sao, nên không dùng accuracy làm metric chính.
- Biểu đồ count plot/bar chart tỷ lệ từng mức review đã được chọn cho phần trực quan: nhóm 5 sao chiếm áp đảo, trong khi nhóm 1–2 sao là nhóm nguy cơ cần quan tâm.

**Bước tiếp theo**

- Q02–Q08: tìm các yếu tố gắn với nhóm 1–2 sao, nhất là giao trễ, độ phức tạp đơn hàng và khu vực logistics.
- Mô hình hóa: ưu tiên nhị phân ≤2 là không hài lòng và ≥4 là hài lòng, hoặc giữ 3 nhóm thấp/trung lập/cao; dùng `class_weight='balanced'`, stratified split, macro-F1, PR-AUC và confusion matrix.
- Ưu tiên stratified split để giữ tỷ lệ nhãn trong tập train/test; nếu bài toán yêu cầu dự báo tương lai, bổ sung kiểm tra chia theo thời gian trên data có thứ tự.


## Q02: Đơn giao trễ so với đúng hạn

**Mục tiêu:** So sánh điểm đánh giá trung bình, tỷ lệ đánh giá tốt và tỷ lệ đánh giá thấp giữa đơn giao trễ và đơn giao đúng hạn hoặc sớm.

**Cách thực hiện:** Nhóm theo `is_late`: `delay_days > 0` là giao trễ, còn `delay_days <= 0` là đúng hạn hoặc sớm. Sau đó tính số đơn, điểm đánh giá trung bình và tỷ lệ đánh giá tốt (≥4) hoặc thấp (≤2).

**Ý nghĩa đối với mô hình:** Nếu đơn trễ có điểm thấp đáng kể hơn, `is_late` là đặc trưng dự báo có khả năng mạnh. Mối quan hệ cũng cho biết mô hình cần ưu tiên hiểu chất lượng giao hàng thay vì xem `review_score` hoàn toàn độc lập với trải nghiệm vận chuyển.

**Mục đích:** Xác định và lượng hóa hiệu ứng nhị phân của giao trễ đối với review score; đồng thời xem đây có phải là đặc trưng phù hợp cho mô hình dự đoán sau giao hàng hay không.

**Kết quả mong đợi:** DataFrame `df_q02` chứa hai nhóm giao hàng cùng các chỉ số đánh giá tương ứng.

In [4]:
q02 = """
SELECT CASE WHEN is_late = 1 THEN 'Giao tre'
            ELSE 'Dung han hoac som' END AS delivery_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY is_late
ORDER BY delivery_group;
"""

df_q02 = pd.read_sql_query(q02, conn)
df_q02


,delivery_group,order_count,avg_review_score,positive_pct,low_score_pct
0,Dung han hoac som,88978,4.29,82.71,9.23
1,Giao tre,6351,2.27,26.72,62.37


## Kết luận và bước tiếp theo — Q02

**Kết luận**

- Đơn giao đúng hạn/sớm có **88.978 đơn**, điểm trung bình **4,29**, tỷ lệ đánh giá tốt **82,71%** và tỷ lệ đánh giá thấp **9,23%**.
- Đơn giao trễ có **6.351 đơn**, điểm trung bình **2,27**, tỷ lệ đánh giá tốt **26,72%** và tỷ lệ đánh giá thấp **62,37%**. Khoảng cách **56 điểm phần trăm đánh giá tốt** cho thấy giao trễ gắn với mức hài lòng kém rõ rệt.
- Biểu đồ phù hợp: boxplot/violin `is_late × review_score` và stacked bar tỷ lệ đánh giá tốt/thấp.

**Bước tiếp theo**

- Xem độ trễ liên tục và các bucket trễ ở Q03 để xác định ngưỡng tác động.
- Đặc trưng `is_late` và `delay_days` chỉ dùng cho mô hình dự đoán `sau khi giao hàng`; không đưa vào mô hình dự đoán tại thời điểm đặt hàng.


## Q03: Mức độ trễ theo nhóm ngày

**Mục tiêu:** Phân tích điểm đánh giá theo từng mức độ trễ giao hàng.

**Cách thực hiện:** Phân loại đơn thành đúng hạn/sớm, trễ 1–3 ngày, trễ 4–7 ngày và trễ trên 7 ngày bằng `CASE`. Sau đó nhóm theo `delay_group`, tính số đơn, điểm trung bình và tỷ lệ đánh giá tốt (≥4).

**Ý nghĩa đối với mô hình:** Phân tích giúp kiểm tra mối quan hệ có tuyến tính với độ trễ hay không, ví dụ mức trễ lớn hơn tương ứng điểm review giảm mạnh hơn. Đây là cơ sở để tạo đặc trưng dạng bucket hoặc giữ nguyên số ngày trễ.

**Mục đích:** Xác định cách biểu diễn `delay_days` hiệu quả nhất và đánh giá liệu số ngày trễ có đủ giá trị dự báo để đưa vào mô hình.

**Kết quả mong đợi:** DataFrame `df_q03` chứa số đơn và mức hài lòng theo từng nhóm trễ.

In [5]:
q03 = """
SELECT CASE
           WHEN delay_days <= 0 THEN 'Som hoac dung han'
           WHEN delay_days <= 3 THEN 'Tre 1-3 ngay'
           WHEN delay_days <= 7 THEN 'Tre 4-7 ngay'
           ELSE 'Tre tren 7 ngay'
       END AS delay_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct
FROM v_review_order_features
WHERE delay_days IS NOT NULL
GROUP BY delay_group
ORDER BY MIN(CASE
                 WHEN delay_days <= 0 THEN 0
                 WHEN delay_days <= 3 THEN 1
                 WHEN delay_days <= 7 THEN 2
                 ELSE 3 END);
"""

df_q03 = pd.read_sql_query(q03, conn)
df_q03


,delay_group,order_count,avg_review_score,positive_pct
0,Som hoac dung han,88978,4.29,82.71
1,Tre 1-3 ngay,1841,3.29,53.23
2,Tre 4-7 ngay,1741,2.11,22.57
3,Tre tren 7 ngay,2769,1.69,11.70


## Kết luận và bước tiếp theo — Q03

**Kết luận**

- Điểm review trung bình giảm từ **4,29** ở nhóm đúng hạn/sớm xuống **3,29** với trễ 1–3 ngày, **2,11** với trễ 4–7 ngày và **1,69** với trễ trên 7 ngày.
- Tỷ lệ đánh giá tốt tương ứng giảm từ **82,71%** xuống **53,23%**, **22,57%** và **11,70%**. Nhóm trễ trên 7 ngày chỉ có **2.769 đơn** nhưng có điểm review thấp nhất.
- Đường bar chart theo bucket trễ cho thấy mối quan hệ âm mạnh; quan hệ này cần kiểm tra cùng các biến khác trước khi coi là nhân quả.

**Bước tiếp theo**

- Thêm độ trễ liên tục và các bucket trễ làm đặc trưng hậu giao hàng.
- Kiểm tra tương tác với bang, phí ship và khoảng cách để xem trễ kéo dài có đồng biến với khu vực logistics hay không.


## Q04: Phí ship và tỷ lệ phí trên giá hàng

**Mục tiêu:** Đánh giá mối liên hệ giữa tổng phí vận chuyển, tỷ lệ phí ship trên giá trị hàng và mức đánh giá.

**Cách thực hiện:** Dùng `NTILE(4)` để chia tổng phí ship và tỷ lệ phí ship/giá hàng thành bốn nhóm từ thấp đến cao, rồi tính giá trị trung bình, điểm review trung bình và tỷ lệ review 1–2 cho mỗi nhóm.

**Ý nghĩa đối với mô hình:** Nếu review score thay đổi nhất quán theo giá trị hoặc tỷ lệ phí ship, chi phí vận chuyển có thể phản ánh chi phí ẩn, chất lượng giao hàng hoặc kỳ vọng giá trị của khách hàng. Tỷ lệ phí ship trên giá hàng thường hữu ích hơn tổng phí vì chuẩn hóa theo quy mô đơn.

**Mục đích:** Xác định đặc trưng chi phí vận chuyển tiềm năng, hỗ trợ biến đổi phi tuyến bằng quartile và tránh giả định tổng phí ship ảnh hưởng review theo cách tuyến tính.

**Kết quả mong đợi:** DataFrame `df_q04` chứa hai chỉ số vận chuyển, phân thành bốn nhóm quartile và các chỉ số review tương ứng.

In [6]:
q04 = """
WITH freight_quartiles AS (
    SELECT *, NTILE(4) OVER (ORDER BY total_freight) AS quartile_number
    FROM v_review_order_features
),
ratio_quartiles AS (
    SELECT *, NTILE(4) OVER (ORDER BY freight_to_price_pct) AS quartile_number
    FROM v_review_order_features
    WHERE freight_to_price_pct IS NOT NULL
),
freight_summary AS (
    SELECT 'Tổng phí ship (BRL)' AS chi_so,
           CASE quartile_number
               WHEN 1 THEN 'Nhóm 1 - 25% phí thấp nhất'
               WHEN 2 THEN 'Nhóm 2 - từ 25%-50%' 
               WHEN 3 THEN 'Nhóm 3 - từ 50%-75%'
               ELSE 'Nhóm 4 - 25% phí cao nhất'
           END AS nhom_quartile,
           COUNT(*) AS so_don,
           ROUND(AVG(total_freight), 2) AS gia_tri_trung_binh,
           ROUND(AVG(review_score), 2) AS diem_review_trung_binh
    FROM freight_quartiles
    GROUP BY quartile_number
),
ratio_summary AS (
    SELECT 'Tỷ lệ phí ship/giá hàng (%)' AS chi_so,
           CASE quartile_number
               WHEN 1 THEN 'Nhóm 1 - 25% tỷ lệ thấp nhất'
               WHEN 2 THEN 'Nhóm 2 - từ 25%-50%'
               WHEN 3 THEN 'Nhóm 3 - từ 50%-75%' 
               ELSE 'Nhóm 4 - 25% tỷ lệ cao nhất'
           END AS nhom_quartile,
           COUNT(*) AS so_don,
           ROUND(AVG(freight_to_price_pct), 2) AS gia_tri_trung_binh,
           ROUND(AVG(review_score), 2) AS diem_review_trung_binh
    FROM ratio_quartiles
    GROUP BY quartile_number
)
SELECT * FROM freight_summary
UNION ALL
SELECT * FROM ratio_summary
ORDER BY chi_so DESC, nhom_quartile;
"""

df_q04 = pd.read_sql_query(q04, conn)
df_q04


,chi_so,nhom_quartile,so_don,gia_tri_trung_binh,diem_review_trung_binh
0,Tỷ lệ phí ship/giá hàng (%),Nhóm 1 - 25% tỷ lệ thấp nhất,23833,8.41,4.20
1,Tỷ lệ phí ship/giá hàng (%),Nhóm 2 - từ 25%-50%,23832,17.55,4.17
2,Tỷ lệ phí ship/giá hàng (%),Nhóm 3 - từ 50%-75%,23832,29.18,4.15
3,Tỷ lệ phí ship/giá hàng (%),Nhóm 4 - 25% tỷ lệ cao nhất,23832,68.18,4.12
4,Tổng phí ship (BRL),Nhóm 1 - 25% phí thấp nhất,23833,10.45,4.32
5,Tổng phí ship (BRL),Nhóm 2 - từ 25%-50%,23832,15.46,4.19
6,Tổng phí ship (BRL),Nhóm 3 - từ 50%-75%,23832,19.78,4.18
7,Tổng phí ship (BRL),Nhóm 4 - 25% phí cao nhất,23832,45.38,3.94


## Kết luận và bước tiếp theo — Q04

**Kết luận**

- Với tổng phí ship, điểm review trung bình giảm nhẹ từ **4,32** ở quartile thấp nhất xuống **3,94** ở quartile cao nhất; phí ship trung bình tương ứng tăng từ **10,45 BRL** lên **45,38 BRL**.
- Với tỷ lệ phí ship/giá hàng, điểm review trung bình giảm từ **4,20** xuống **4,12** khi giá trị trung bình của tỷ lệ tăng từ **8,41%** lên **68,18%**.
- Tổng phí ship có phân tách hơn so với tỷ lệ phí/gía hàng, nhưng khác biệt điểm vẫn nhỏ nên cần kiểm tra phương sai và các biến khu vực trước khi kết luận.

**Bước tiếp theo**

- Vẽ bar chart theo quartile và scatter giữa phí ship/tỷ lệ phí ship với review.
- Kiểm tra tương quan giữa phí ship, giá trị đơn và khoảng cách địa lý để tránh giải thích nhầm do đơn hàng xa hoặc giá trị thấp.


## Q05: Độ phức tạp của đơn hàng

**Mục tiêu:** So sánh mức hài lòng theo số mặt hàng và số người bán trong đơn.

**Cách thực hiện:** Nhóm đơn theo số mặt hàng (1; 2–3; 4 trở lên) và số người bán (1 người bán hoặc nhiều người bán). Với mỗi tổ hợp, tính số đơn, điểm review trung bình và tỷ lệ đánh giá 1–2.

**Ý nghĩa đối với mô hình:** Số mặt hàng và số người bán có thể đại diện cho độ phức tạp giao dịch, khả năng tách gói hàng và rủi ro giao hàng. Nếu tổ hợp nhiều mặt hàng/nhiều người bán có tỷ lệ 1–2 sao cao hơn, các đặc trưng này có thể tăng khả năng dự đoán review tiêu cực.

**Mục đích:** Đánh giá tác dụng của `item_count`, `seller_count` và tương tác giữa chúng, từ đó cân nhắc tạo đặc trưng basket complexity hoặc multi-seller order.

**Kết quả mong đợi:** DataFrame `df_q05` chứa mức hài lòng theo từng nhóm độ phức tạp đơn hàng.

In [7]:
q05 = """
SELECT CASE
           WHEN item_count = 1 THEN '1 mat hang'
           WHEN item_count BETWEEN 2 AND 3 THEN '2-3 mat hang'
           ELSE '4+ mat hang'
       END AS basket_group,
       CASE WHEN seller_count = 1 THEN '1 nguoi ban'
            ELSE 'Nhieu nguoi ban' END AS seller_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY basket_group, seller_group
ORDER BY MIN(item_count), MIN(seller_count);
"""

df_q05 = pd.read_sql_query(q05, conn)
df_q05


,basket_group,seller_group,order_count,avg_review_score,low_score_pct
0,1 mat hang,1 nguoi ban,85857,4.21,11.24
1,2-3 mat hang,1 nguoi ban,7417,3.78,22.83
2,2-3 mat hang,Nhieu nguoi ban,1134,2.90,46.12
3,4+ mat hang,1 nguoi ban,807,3.49,30.48
4,4+ mat hang,Nhieu nguoi ban,114,2.49,59.65


## Kết luận và bước tiếp theo — Q05

**Kết luận**

- Đơn có 1 mặt hàng và 1 người bán có **85.857 đơn**, điểm trung bình **4,21** và tỷ lệ 1–2 sao **11,24%**.
- Đơn 2–3 mặt hàng từ nhiều người bán tăng mức thấp xuống **2,90 điểm**, với **46,12%** là 1–2 sao.
- Đơn 4+ mặt hàng từ nhiều người bán có điểm trung bình thấp nhất là **2,49** và tỷ lệ 1–2 sao cao nhất là **59,65%**. Số mẫu chỉ **114 đơn** nên kết quả cần thận trọng.

**Bước tiếp theo**

- Vẽ heatmap/bar chart cho tổ hợp số mặt hàng và số người bán; báo kèm số đơn để không đánh giá mạnh dựa trên nhóm nhỏ.
- Thêm tương tác `item_count × seller_count`, đồng thời kiểm tra delivery delay trước khi coi độ phức tạp là nguyên nhân.


## Q06: Thanh toán và trả góp

**Mục tiêu:** Phân tích ảnh hưởng của hình thức thanh toán và số kỳ trả góp đến mức đánh giá.

**Cách thực hiện:** Nhóm theo phương thức thanh toán đầu tiên để tính số đơn, số kỳ trả góp trung bình, điểm review trung bình và tỷ lệ đánh giá thấp (≤2). Đồng thời có thể chia số kỳ trả góp thành các bucket để so sánh mức hài lòng.

**Ý nghĩa đối với mô hình:** Hình thức thanh toán và số kỳ trả góp có thể phản ánh thu nhập, kênh mua sắm, quy mô đơn hoặc thói quen tài chính. Nếu mức review khác biệt có ý nghĩa, các cột này là đặc trưng phân loại có khả năng dự báo, nhưng không nên hiểu quan hệ quan sát là nhân quả.

**Mục đích:** Kiểm tra giá trị dự báo của thông tin thanh toán; đánh giá có giữ kiểu categorical cho `first_payment_type`, mã hóa `max_installments`, hoặc gom nhóm số kỳ trả góp trước khi huấn luyện mô hình.

**Kết quả mong đợi:** DataFrame `df_q06` chứa số đơn và các chỉ số review theo phương thức thanh toán hoặc nhóm kỳ trả góp.

In [8]:
q06 = """
SELECT first_payment_type AS payment_type,
       COUNT(*) AS order_count,
       ROUND(AVG(max_installments), 2) AS avg_installments,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
WHERE first_payment_type IS NOT NULL
GROUP BY first_payment_type
ORDER BY order_count DESC;
"""

df_q06 = pd.read_sql_query(q06, conn)
df_q06


,payment_type,order_count,avg_installments,avg_review_score,low_score_pct
0,credit_card,73385,3.5,4.16,12.84
1,boleto,18970,1.0,4.16,12.51
2,voucher,1473,1.0,4.11,14.32
3,debit_card,1424,1.0,4.25,10.88


## Kết luận và bước tiếp theo — Q06

**Kết luận**

- Thẻ ghi nợ có điểm trung bình cao nhất **4,25** và tỷ lệ 1–2 sao **10,88%**, trong khi voucher có **4,11** điểm và **14,32%** đánh giá thấp.
- Thẻ tín dụng chiếm **73.385 đơn** nhưng điểm trung bình **4,16** và tỷ lệ 1–2 sao **12,84%**, gần bằng boleto với **18.970 đơn** và **12,51%**.
- Số kỳ trả góp trung bình của thẻ tín dụng là **3,5**, cao hơn các phương thức còn lại ở mức 1 kỳ; khác biệt mức hài lòng tổng thể không lớn.
- Voucher có tỷ lệ 1–2 sao cao nhất nhưng không nhất thiết là nguyên nhân trực tiếp làm review thấp; kết quả này chỉ cho thấy nhóm voucher có mức hài lòng thấp hơn trong dữ liệu quan sát.
- Ngoài ra, truy vấn chỉ dùng `first_payment_type`, nên một đơn thanh toán hỗn hợp có thể được xếp vào nhóm voucher dù còn thẻ tín dụng hoặc phương thức khác. Cần kiểm tra tiếp giao hàng trễ, giá trị đơn và loại sản phẩm trước khi kết luận voucher liên quan xấu đến review.
- Mẫu voucher có **1.473 đơn**, nhỏ hơn rất nhiều so với thẻ tín dụng, vì vậy kết luận về nhóm này cần thận trọng và không nên coi chênh lệch 3,44 điểm phần trăm là ảnh hưởng nhân quả.

**Bước tiếp theo**

- Vẽ payment mix và stacked bar review theo phương thức thanh toán, đồng thời phân nhóm số kỳ trả góp.
- Đối chiếu kết quả với giá trị đơn, số mặt hàng và phí ship để kiểm tra nhóm thanh toán có phải cổng vào cho điều kiện đơn khác hay không.


## Q07: Xu hướng review theo tháng

**Mục tiêu:** Quan sát biến động điểm đánh giá trung bình và tỷ lệ đánh giá theo từng tháng.

**Cách thực hiện:** Nhóm dữ liệu theo `purchase_year_month`, tính số đơn, điểm review trung bình, tỷ lệ đánh giá tốt (≥4) và tỷ lệ đánh giá thấp (≤2). Sau đó sắp xếp kết quả theo năm-tháng tăng dần.

**Ý nghĩa đối với mô hình:** Xu hướng theo thời gian giúp phát hiện mùa vụ, thay đổi chất lượng dịch vụ hoặc hành vi đánh giá theo từng giai đoạn. Sự biến động lớn có thể khiến mô hình học sai xu hướng nếu không chia train/test theo thời gian.

**Mục đích:** Đánh giá khả năng tạo đặc trưng thời gian như năm, tháng hoặc seasonality; đồng thời xác định chiến lược chia dữ liệu time-based để mô hình không học từ tương lai hoặc leak thời gian.

**Kết quả mong đợi:** DataFrame `df_q07` chứa chuỗi 20 tháng dữ liệu cùng các chỉ số mức hài lòng.
***bắt đầu phân tích từ tháng 1 2017

In [9]:
q07 = """
SELECT purchase_year_month,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features

GROUP BY purchase_year_month
ORDER BY purchase_year_month;
"""

df_q07 = pd.read_sql_query(q07, conn)
df_q07


,purchase_year_month,order_count,avg_review_score,positive_pct,low_score_pct
0,2016-10,260,4.01,75.38,16.54
1,2016-12,1,5.00,100.00,0.00
2,2017-01,729,4.20,79.84,11.93
3,2017-02,1634,4.21,79.44,11.02
4,2017-03,2506,4.19,79.93,11.13
5,2017-04,2276,4.13,78.08,13.01
6,2017-05,3490,4.24,80.97,10.00
7,2017-06,3083,4.22,80.60,10.70
8,2017-07,3809,4.26,81.46,9.90
9,2017-08,4127,4.32,83.81,9.33


## Kết luận và bước tiếp theo — Q07

**Kết luận**

- Trong chuỗi dữ liệu, điểm review trung bình tăng từ **3,81** vào tháng 3/2018 lên **4,32** vào tháng 7/2018; tháng 2/2018 cũng chỉ đạt **3,88** với tỷ lệ đánh giá thấp **19,26%**.
- Tháng 11/2017 có **7.195 đơn** nhưng điểm trung bình chỉ **3,99**, so với đỉnh **4,32** ở tháng 8/2017 và tháng 7–8/2018.
- Tháng có ít dữ liệu như 12/2016 chỉ **1 đơn**, nên không diễn giải mức 5,00 điểm là xu hướng chung.

**Bước tiếp theo**

- Vẽ line chart, rolling average và tỷ lệ đánh giá tốt/thấp theo tháng để nhận diện mùa vụ và đỉnh giảm/ tăng.
- Kiểm tra dữ liệu tháng 11/2017 và 2–3/2018 cùng khối lượng đơn, trễ giao và phí ship trước khi dùng biến thời gian trong mô hình.


## Q08: Hài lòng theo bang/khu vực

**Mục tiêu:** So sánh mức hài lòng và chất lượng giao hàng theo bang của khách hàng.

**Cách thực hiện:** `GROUP BY customer_state`, chỉ giữ bang có ít nhất 100 đơn bằng `HAVING COUNT(*) >= 100`, sau đó tính số đơn, điểm trung bình, tỷ lệ đánh giá tốt (≥4), tỷ lệ đánh giá thấp (≤2), tỷ lệ giao trễ và số ngày giao trung bình.

**Ý nghĩa đối với mô hình:** Khác biệt điểm review giữa các bang có thể xuất phát từ chất lượng logistics, khoảng cách vận chuyển, mật độ đơn hàng hoặc đặc điểm khu vực. Đặc trưng bang cần mã hóa phù hợp, đồng thời tránh đưa bang có ít dữ liệu vào khi mẫu không đủ tin cậy.

**Mục đích:** Xác định giá trị dự báo của vị trí khách hàng, đánh giá tương quan giữa khu vực và giao trễ, rồi quyết định giữ `customer_state`, gom khu vực hoặc loại bỏ đặc trưng.

**Kết quả mong đợi:** DataFrame `df_q08` chứa các chỉ số review và logistics cho từng bang đủ mẫu.

In [10]:
q08 = """
SELECT customer_state,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct,
       ROUND(AVG(is_late) * 100.0, 2) AS late_rate_pct,
       ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM v_review_order_features
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY avg_review_score ASC, low_score_pct DESC;
"""

df_q08 = pd.read_sql_query(q08, conn)
df_q08


,customer_state,order_count,avg_review_score,positive_pct,low_score_pct,late_rate_pct,avg_delivery_days
0,MA,706,3.82,69.97,20.11,17.28,21.43
1,AL,393,3.85,71.76,21.12,20.61,24.41
2,SE,332,3.90,72.29,18.98,15.06,21.45
3,PA,924,3.92,72.40,17.86,10.82,23.63
4,BA,3213,3.93,72.92,16.99,11.89,19.24
5,CE,1272,3.94,72.80,17.14,13.68,21.22
6,RJ,12136,3.97,73.48,18.27,11.95,15.24
7,PI,470,3.99,75.11,16.17,13.62,19.48
8,PB,510,4.07,75.88,14.90,10.20,20.28
9,PE,1573,4.08,77.24,14.88,9.47,18.39


## Kết luận và bước tiếp theo — Q08

**Kết luận**

- Trong các bang có ít nhất 100 đơn, MA có điểm review thấp nhất **3,82**, tỷ lệ đánh giá thấp **20,11%** và tỷ lệ giao trễ **17,28%**.
- AL có điểm **3,85**, tỷ lệ trễ cao nhất **20,61%** và thời gian giao trung bình **24,41 ngày**, cho thấy vấn đề logistics có thể trầm trọng hơn ở bang này.
- SP có **40.063 đơn**, điểm **4,25**, tỷ lệ giao trễ **4,44%**, thời gian giao **8,74 ngày**, là bang có quy mô lớn nhất và dịch vụ giao hàng tốt hơn nhóm đầu bảng.

**Bước tiếp theo**

- Vẽ choropleth Brazil, ranking bang và scatter giữa tỷ lệ trễ/thời gian giao với điểm review để ưu tiên khu vực logistics cần cải thiện.
- Thêm `customer_state` vào mô hình bằng mã hóa phân loại chỉ khi có ý nghĩa dự báo và tránh overfit các bang có ít dữ liệu.


# Bảng tổng hợp cho bước khám phá bằng Python

Bảng dưới đây chuyển kết quả SQL thành đầu việc trực quan hóa và khám phá hành vi mua sắm bằng Python: bản đồ khách hàng, phương thức thanh toán và tương quan.

| Truy vấn | Output SQL đã có | Trực quan Python phù hợp | Giá trị cho bước tiếp theo |
|---|---|---|---|
| Q01 | Số lượng và tỷ lệ từng mức review | Count plot/bar chart tỷ lệ 1-5 | Chọn biểu đồ, stratified split và metric khi mô hình hóa |
| Q02 | Review theo đúng hạn/sớm và trễ | Violin/boxplot và stacked bar | Kiểm tra tác động giao trễ; biến hậu giao hàng |
| Q03 | Review theo bucket mức độ trễ | Line/bar chart theo bucket | Xác định ngưỡng trễ ảnh hưởng mạnh nhất |
| Q04 | Quartile phí ship và tỷ lệ phí ship | Bar chart theo quartile, scatter và correlation | Kiểm tra chi phí ship liên quan đến review và giá trị đơn |
| Q05 | Review theo số item, sản phẩm và seller | Heatmap/bar chart độ phức tạp | Phân tích đơn phức tạp và tạo feature tương tác |
| Q06 | Payment type, installment và review | Payment mix, stacked bar và histogram | Trực quan hóa hành vi thanh toán theo nhóm khách hàng |
| Q07 | Review và số đơn theo tháng | Line chart 22 tháng và rolling average | Phát hiện xu hướng, mùa vụ và thời điểm mua sắm |
| Q08 | Review, trễ và thời gian giao theo bang | Choropleth Brazil, ranking và scatter | Xác định vùng logistics cần cải thiện trên bản đồ |

## Hướng dùng cho EDA Python
- Ưu tiên trực quan các yếu tố có khác biệt lớn: giao trễ, mức độ trễ, độ phức tạp và khu vực.
- Dùng bản đồ bang/khu vực kết hợp tỷ lệ giao trễ và điểm review thấp để tìm vị trí cần tối ưu logistics.
- Dùng phương thức thanh toán và số kỳ trả góp để phân tích hành vi thanh toán theo nhóm khách hàng.
- Dùng heatmap tương quan và scatter plot cho phí ship, giá trị đơn, khoảng cách, thời gian giao và review.
- Không dùng đặc trưng sau giao hàng như delay_days, is_late, delivery_days cho mô hình dự đoán tại thời điểm đặt hàng.


In [11]:
# Đóng kết nối sau khi hoàn tất toàn bộ truy vấn.
conn.close()
